# Busca semantica de cartas (Etapa 3.2)

*(Feito com ClaudeAI: https://claude.ai/share/b3536ad5-de2f-4b3f-a788-ba1c9a17f7da)*

Usa os embeddings gerados na etapa 3.1 pra montar uma funcao de busca de
verdade: consulta em linguagem natural -> vetor -> similaridade de
cosseno -> top-k cartas, com nome/tipo/custo/texto formatados pra leitura,
nao so uma lista crua de scores.

Basta ajustar os caminhos na celula de configuracao e rodar tudo
(`Run All`). A ultima celula e' interativa -- digite sua propria consulta.

## Configuração

In [1]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer

meta = json.load(open("Dados/Etapa2_embeddings/embeddings_qwen3_meta.json", encoding="utf-8"))

CARD_DOCUMENTS_PATH = "Dados/Etapa1_card_expanded/card_documents.jsonl"
EMBEDDINGS_PATH = "Dados/Etapa2_embeddings/embeddings_qwen3.npy"
MODEL_NAME = meta["model_id"]           # "Qwen/Qwen3-Embedding-0.6B" (ou a pasta local, se estiver offline)
QUERY_PREFIX = meta["query_prefix"]     # "Instruct: ...\nQuery:"

c:\Users\joaquim25018\.conda\envs\Token\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Carregar embeddings e dados completos da carta

Os embeddings (etapa 3.1) foram gerados na mesma ordem em que
`card_documents.jsonl` lista as cartas -- filtrando so as que tem
`embedding_text_structured` (mesmo criterio usado pra gerar os vetores).
Reconstruimos essa lista de nomes direto do arquivo, sem depender de um
segundo arquivo de metadados separado (mais facil de esquecer de enviar
junto e desnecessario, ja que a ordem e' sempre a mesma).

In [2]:
embeddings = np.load(EMBEDDINGS_PATH)

cards = []
with open(CARD_DOCUMENTS_PATH, encoding="utf-8") as f:
    for line in f:
        card = json.loads(line)
        if not card.get("embedding_text"):
            continue
        cards.append(card)

assert len(cards) == embeddings.shape[0], (
    f"Descompasso: {len(cards)} cartas vs {embeddings.shape[0]} vetores -- "
    "regere os embeddings (etapa 3.1)."
)

# a search() antiga ainda usa estes dois:
names = [c["name"] for c in cards]
cards_by_name = {}
for c in cards:
    cards_by_name.setdefault(c["name"], c)

print(f"{len(cards)} vetores carregados, shape {embeddings.shape}")

33147 vetores carregados, shape (33147, 1024)


In [3]:
from filtros_mtg import set_metadata_indexes, _KEYWORD_INDEX, _TAG_INDEX
set_metadata_indexes(cards)
print(f"{len(_KEYWORD_INDEX)} keywords e {len(_TAG_INDEX)} tags indexadas")
print(sorted(cards[0].keys()))

0 keywords e 0 tags indexadas
['colors', 'defense', 'embedding_text', 'faces', 'keywords', 'loyalty', 'mana_cost', 'name', 'oracle_text', 'otags', 'power', 'produced_mana', 'toughness', 'type_line']


## Carregar o modelo

Mesmo modelo da etapa 3.1 -- precisa ser o mesmo, porque vetores gerados
por modelos diferentes nao sao comparaveis entre si.

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL_NAME, processor_kwargs={"padding_side": "left"})
model.max_seq_length = meta["max_seq_length"]
print(f"Modelo carregado: {MODEL_NAME}")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 5686.57it/s]


Modelo carregado: Qwen/Qwen3-Embedding-0.6B


## Funcao de busca

`search(query, k)` embeda a consulta (com o prefixo que os modelos BGE
esperam pra busca -- documentado na etapa 3.1), calcula similaridade de
cosseno contra os 33 mil vetores de uma vez (produto de matriz, rapido o
suficiente sem precisar de indice tipo FAISS nesse tamanho de dataset) e
devolve os top-k com os dados completos da carta.

In [5]:
def search(query: str, k: int = 10):
    query_vec = model.encode([QUERY_PREFIX + query], normalize_embeddings=True)[0]
    scores = embeddings @ query_vec
    top_k = np.argsort(-scores)[:k]

    results = []
    for i in top_k:
        name = names[i]
        card = cards_by_name.get(name)
        if card is None:
            continue
        results.append({
            "score": float(scores[i]),
            "name": name,
            "type_line": card.get("type_line"),
            "mana_cost": card.get("mana_cost"),
            "oracle_text": card.get("oracle_text"),
        })
    return results


def print_results(results):
    for r in results:
        score = f"{r['score']:.3f}" if r["score"] is not None else "  -  "
        print(f"{score}  {r['name']}  ({r['mana_cost'] or '-'})")
        print(f"       {r['type_line']}")
        if r["oracle_text"]:
            print(f"       {r['oracle_text'][:120]}")
        print()

In [6]:
from filtros_mtg import hybrid_search

def search2(query, k=10):
    return hybrid_search(query, model, embeddings, cards, QUERY_PREFIX, k=k, embed_full_query=False)

In [7]:
import os, getpass
from openai import OpenAI         
from filtros_mtg import parse_query_llm

token = os.environ.get("ILUMA_TOKEN") or getpass.getpass("Token da IlumA (sk-...): ")
cliente = OpenAI(base_url="https://iluma.cnpem.br:4000/v1", api_key=token,
                 timeout=300.0, max_retries=1)

def search3(query, k=10):
    return hybrid_search(query, model, embeddings, cards, QUERY_PREFIX, k=k,
                         parser=lambda q: parse_query_llm(q, cliente, model="iluma"),
                         embed_full_query=False)


## Testando com algumas consultas

In [8]:
# for query in [
#     "creature that destroys artifacts",
#     "cheap green mana creature",
#     "counter target spell",
#     "deals damage to any target",
# ]:
#     print(f"\n=== BASE: {query!r} ===")
#     print_results(search(query, k=5))
#     print(f"=== HIBRIDO: {query!r} ===")
#     print_results(search2(query, k=5))
#     print(f"=== HIBRIDO+ILUMA: {query!r} ===")
#     print_results(search3(query, k=5))

## Sua consulta

Troque o texto abaixo e rode a celula quantas vezes quiser.

In [9]:
print_results(search3("instant that draws 2 cards and creates 2 tokens artifacts", k=10))

filtros: {'types': ['instant']} | texto semantico: 'draws 2 cards and creates 2 artifact tokens' | 3830/33147 cartas passam
0.694  Masterful Replication  ({5}{U})
       Instant
       Choose one —
• Create two 3/3 colorless Golem artifact creature tokens.
• Choose target artifact you control. Each other

0.691  Artificer's Epiphany  ({2}{U})
       Instant
       Draw two cards. If you control no artifacts, discard a card.

0.678  Glimmerburst  ({3}{U})
       Instant
       Draw two cards. Create a 1/1 white Glimmer enchantment creature token.

0.673  Depose // Deploy  ({1}{W/U} // {2}{W}{U})
       Instant // Instant
       Tap target creature.
Draw a card. // Create two 1/1 colorless Thopter artifact creature tokens with flying, then you gai

0.666  Stern Lesson  ({2}{U})
       Instant
       Draw two cards, then discard a card. Create a tapped Powerstone token. (It's an artifact with "{T}: Add {C}. This mana c

0.665  Vision of Love  ({1}{R})
       Instant
       You may sacrifi

In [10]:
for query in [
    "dinosaur that costs 7 mana and is card advantage",
    "removal that sacrifices a creature I control and generates card advantage"
]:
    print(f"\n=== PESQUISA: {query!r} ===")
    print_results(search3(query, k=5))


=== PESQUISA: 'dinosaur that costs 7 mana and is card advantage' ===
filtros: {'cmc_min': 7, 'cmc_max': 7, 'tags': ['card-advantage']} | texto semantico: 'dinosaur' | 163/33147 cartas passam
0.473  Treeshaker Chimera  ({5}{G}{G})
       Creature — Chimera
       All creatures able to block this creature do so.
When this creature dies, draw three cards.

0.447  Regal Force  ({4}{G}{G}{G})
       Creature — Elemental
       When this creature enters, draw a card for each green creature you control.

0.445  Spawnbed Protector  ({7})
       Creature — Eldrazi
       At the beginning of your end step, return up to one target Eldrazi creature card from your graveyard to your hand. Creat

0.431  Skyline Despot  ({5}{R}{R})
       Creature — Dragon
       Flying
When this creature enters, you become the monarch.
At the beginning of your upkeep, if you're the monarch, create

0.410  Hollow Marauder  ({6}{B})
       Creature — Specter Rogue
       This spell costs {1} less to cast for each crea